In [0]:
from pyspark.sql.functions import *

DataBase = "workspace.ecommerce"



In [0]:
orders = (
    spark.table(f"{DataBase}.bronze_orders")
    .selectExpr("order_id",
                "customer_id",
                "order_status", 
                "try_cast(order_purchase_timestamp as timestamp) as purchase_ts",
                "try_cast(order_delivered_customer_date as timestamp) as delivered_ts",
                "try_cast(order_estimated_delivery_date as timestamp) as estimated_ts")
    .dropDuplicates(["order_id"])
    .filter("order_id is not null and customer_id is not null")            
)


customers = (
    spark.table(f"{DataBase}.bronze_customers")
    .selectExpr("customer_id",
                "customer_unique_id",
                "initcap(customer_city) as city",
                "upper(customer_state) as state"
                )
    .dropDuplicates(["customer_id"])
)

sellers = (
    spark.table(f"{DataBase}.bronze_sellers")
    .selectExpr("seller_id",
                "initcap(seller_city) as city",
                "upper(seller_state) as state")
    .dropDuplicates(["seller_id"])
)

In [0]:
items = (
    spark.table(f"{DataBase}.bronze_order_items")
         .selectExpr(
             "order_id",
             "try_cast(order_item_id as int) as order_item_id",
             "product_id",
             "seller_id",
             "try_cast(price as double) as price",
             "try_cast(freight_value as double) as freight_value"
         )
         .dropDuplicates(["order_id", "order_item_id"])
         .filter("order_id is not null and price >= 0")
)

payments = (
    spark.table(f"{DataBase}.bronze_order_payments")
         .selectExpr(
             "order_id",
             "try_cast(payment_sequential as int) as payment_sequential",
             "payment_type",
             "try_cast(payment_installments as int) as installments",
             "try_cast(payment_value as double) as payment_value"
         )
         .dropDuplicates(["order_id", "payment_sequential"])
         .filter("order_id is not null and payment_value >= 0")
)

In [0]:
reviews = (
    spark.table(f"{DataBase}.bronze_order_reviews")
         .selectExpr(
             "review_id",
             "order_id",
             "try_cast(review_score as int) as review_score",
             "try_cast(review_answer_timestamp as timestamp) as answered_ts"
         )
         .filter("review_score between 1 and 5")
         .withColumn("rn", expr("row_number() over (partition by order_id order by answered_ts desc)"))
         .filter("rn = 1")
         .drop("rn")
)

In [0]:
translation = spark.table(f"{DataBase}.bronze_category_translation")

products = (
    spark.table(f"{DataBase}.bronze_products")
    .join(translation, "product_category_name", "left")
    .selectExpr("product_id",
                "coalesce(product_category_name_english, product_category_name, 'unknown') as category")
    .dropDuplicates(["product_id"])
)

In [0]:
order_details = (
    items.join(orders, "order_id", "inner")
    .join(customers, "customer_id", "left")
    .join(products, "product_id", "left")
    .withColumn("purchase_month", expr("date_format(purchase_ts, 'yyyy-MM')"))
    .withColumn("delivery_days", expr("datediff(delivered_ts, purchase_ts)"))
    .withColumn("is_late", expr("case when delivered_ts is null then null when delivered_ts > estimated_ts then 1 else 0 end"))
)

orders.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{DataBase}.silver_orders")
customers.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{DataBase}.silver_customers")
sellers.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{DataBase}.silver_sellers")
items.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{DataBase}.silver_order_items")
payments.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{DataBase}.silver_payments")
reviews.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{DataBase}.silver_reviews")
products.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{DataBase}.silver_products")
order_details.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{DataBase}.silver_order_details")

In [0]:
raw_orders = spark.table(f"{DataBase}.bronze_orders").count()
clean_orders = spark.table(f"{DataBase}.silver_orders").count()
print("raw:", raw_orders, "clean:", clean_orders, "removed:", raw_orders - clean_orders)

silver_order_details = spark.table(f"{DataBase}.silver_order_details")
silver_order_details.show(5, False)
